In [ ]:
import base64, hashlib, json, os, subprocess, sys, tempfile
from pathlib import Path
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in ('1','true'):
    raise RuntimeError('COMPETITION_RERUN_PROHIBITED')
assets = {'runner.py': {'sha256': '08891bda91b86e80b36e0e1bad768a0eccddae313b26d726689c41e8943591ed', 'base64': 'IiIiVHJ1c3RlZCBzdGVyaWxlIGVudHJ5cG9pbnQuIE5vIGFzc2Vzc29yLCBoaWRkZW4gdmFsdWVzIG9yIHByb3ZpZGVyIGNyZWRlbnRpYWxzLgpMaW51eCB4ODYtNjQgb25seS4gQWxsIGNhbmRpZGF0ZSBleGVjdXRpb24gZm9sbG93cyBzZWFsKCk7IG5vIGZhbGxiYWNrLgpQeXRob24gc3RhcnRzIGJlZm9yZSBjaHJvb3Q7IG9ubHkgZXhwbGljaXQgcnVudGltZSBmaWxlcyBzdXBwb3J0IGxhdGVyIGltcG9ydHMuCiIiIgppbXBvcnQgY3R5cGVzCmltcG9ydCBlcnJubwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCByZXNvdXJjZQppbXBvcnQgc2lnbmFsCmltcG9ydCBzb2NrZXQKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkxJQkMgPSBjdHlwZXMuQ0RMTChOb25lLCB1c2VfZXJybm89VHJ1ZSkKIyB4ODYtNjQgbmF0aXZlIEFCSSBvbmx5OyBhbGwgdW5rbm93biBzeXNjYWxscyAoaW5jbHVkaW5nIHgzMikgcmV0dXJuIEVQRVJNLgojIE5vIHNvY2tldCwgcHJvY2VzcyBjcmVhdGlvbi9leGVjLCBwcm9jZXNzIGluc3BlY3Rpb24sIHNpZ25hbHMsIGZpbGVzeXN0ZW0KIyBtdXRhdGlvbiwgbmFtZXNwYWNlcywgcHJpdmlsZWdlIGNoYW5nZXMsIGlvX3VyaW5nLCBicGYgb3Iga2V5cmluZyBjYWxscy4KQUxMT1dFRCA9IHswLCAxLCAyLCAzLCA1LCA4LCA5LCAxMCwgMTEsIDEyLCAxMywgMTQsIDE1LCAxNywgMTksIDIwLAogICAgICAgICAgIDIxLCAyNCwgMjUsIDI4LCAzNSwgMzksIDYwLCA2MywgNzksIDg5LCA5NywgMTAyLCAxMDQsCiAgICAgICAgICAgMTA3LCAxMDgsIDExMCwgMTE1LCAxMTgsIDEyMCwgMTU4LCAxODYsIDIwMiwgMjE4LAogICAgICAgICAgIDIyOCwgMjI5LCAyMzAsIDIzMSwgMjU3LCAyNjIsIDI2NywgMjczLCAzMTgsIDMzMn0KCmNsYXNzIEZpbHRlcihjdHlwZXMuU3RydWN0dXJlKToKICAgIF9maWVsZHNfID0gWygnY29kZScsIGN0eXBlcy5jX3VzaG9ydCksICgnanQnLCBjdHlwZXMuY191Ynl0ZSksCiAgICAgICAgICAgICAgICAoJ2pmJywgY3R5cGVzLmNfdWJ5dGUpLCAoJ2snLCBjdHlwZXMuY191aW50KV0KY2xhc3MgUHJvZ3JhbShjdHlwZXMuU3RydWN0dXJlKToKICAgIF9maWVsZHNfID0gWygnbGVuZ3RoJywgY3R5cGVzLmNfdXNob3J0KSwgKCdmaWx0ZXJzJywgY3R5cGVzLlBPSU5URVIoRmlsdGVyKSldCmNsYXNzIENhcEhlYWRlcihjdHlwZXMuU3RydWN0dXJlKToKICAgIF9maWVsZHNfID0gWygndmVyc2lvbicsIGN0eXBlcy5jX3VpbnQpLCAoJ3BpZCcsIGN0eXBlcy5jX2ludCldCmNsYXNzIENhcERhdGEoY3R5cGVzLlN0cnVjdHVyZSk6CiAgICBfZmllbGRzXyA9IFsoJ2VmZmVjdGl2ZScsIGN0eXBlcy5jX3VpbnQpLCAoJ3Blcm1pdHRlZCcsIGN0eXBlcy5jX3VpbnQpLAogICAgICAgICAgICAgICAgKCdpbmhlcml0YWJsZScsIGN0eXBlcy5jX3VpbnQpXQoKZGVmIHByY3RsKG9wdGlvbiwgYT0wLCBiPTAsIGM9MCwgZD0wKToKICAgIHJlc3VsdCA9IExJQkMucHJjdGwoY3R5cGVzLmNfaW50KG9wdGlvbiksIGN0eXBlcy5jX3Vsb25nKGEpLAogICAgICAgICAgICAgICAgICAgICAgICBjdHlwZXMuY191bG9uZyhiKSwgY3R5cGVzLmNfdWxvbmcoYyksIGN0eXBlcy5jX3Vsb25nKGQpKQogICAgaWYgcmVzdWx0IDwgMDoKICAgICAgICByYWlzZSBPU0Vycm9yKGN0eXBlcy5nZXRfZXJybm8oKSwgJ3ByY3RsICVzJyAlIG9wdGlvbikKICAgIHJldHVybiByZXN1bHQKCmRlZiBxdWFsaWZ5KCk6CiAgICBpZiBzeXMucGxhdGZvcm0gIT0gJ2xpbnV4JyBvciBwbGF0Zm9ybS5tYWNoaW5lKCkgIT0gJ3g4Nl82NCcgb3IgY3R5cGVzLnNpemVvZihjdHlwZXMuY192b2lkX3ApICE9IDg6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdVTlNVUFBPUlRFRF9QTEFURk9STScpCiAgICBpZiBvcy5nZXRldWlkKCkgIT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1JPT1RfUkVRVUlSRUQnKQogICAgaWYgbm90IGhhc2F0dHIob3MsICdjaHJvb3QnKSBvciBub3QgaGFzYXR0cihvcywgJ3NldHJlc3VpZCcpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignTUlTU0lOR19JREVOVElUWV9QUklNSVRJVkUnKQoKZGVmIHNlY2NvbXAoKToKICAgIGluc3RydWN0aW9ucyA9IFsoMHgyMCwgMCwgMCwgNCksICgweDE1LCAxLCAwLCAweEMwMDAwMDNFKSwKICAgICAgICAgICAgICAgICAgICAoMHgwNiwgMCwgMCwgMHg4MDAwMDAwMCksICgweDIwLCAwLCAwLCAwKV0KICAgIGZvciBudW1iZXIgaW4gc29ydGVkKEFMTE9XRUQpOgogICAgICAgIGluc3RydWN0aW9ucyArPSBbKDB4MTUsIDAsIDEsIG51bWJlciksICgweDA2LCAwLCAwLCAweDdGRkYwMDAwKV0KICAgIGluc3RydWN0aW9ucyArPSBbKDB4MDYsIDAsIDAsIDB4MDAwNTAwMDAgfCBlcnJuby5FUEVSTSldCiAgICBmaWx0ZXJzID0gKEZpbHRlciAqIGxlbihpbnN0cnVjdGlvbnMpKSgqKEZpbHRlcigqeCkgZm9yIHggaW4gaW5zdHJ1Y3Rpb25zKSkKICAgIHByb2dyYW0gPSBQcm9ncmFtKGxlbihpbnN0cnVjdGlvbnMpLCBmaWx0ZXJzKQogICAgcHJjdGwoMzgsIDEpCiAgICBpZiBwcmN0bCgzOSkgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05PX05FV19QUklWU19OT1RfU0VUJykKICAgIGlmIExJQkMucHJjdGwoMjIsIDIsIGN0eXBlcy5ieXJlZihwcm9ncmFtKSwgMCwgMCkgIT0gMDoKICAgICAgICByYWlzZSBPU0Vycm9yKGN0eXBlcy5nZXRfZXJybm8oKSwgJ3NlY2NvbXAgaW5zdGFsbCcpCgpkZWYgYXJtX3BhcmVudF9kZWF0aChwYXJlbnRfcGlkKToKICAgIHByY3RsKDEsIHNpZ25hbC5TSUdLSUxMKQogICAgb2JzZXJ2ZWQgPSBjdHlwZXMuY19pbnQoKQogICAgaWYgTElCQy5wcmN0bCgyLCBjdHlwZXMuYnlyZWYob2JzZXJ2ZWQpLCAwLCAwLCAwKSAhPSAwIG9yIG9ic2VydmVkLnZhbHVlICE9IHNpZ25hbC5TSUdLSUxMOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignUERFQVRIU0lHX05PVF9TRVQnKQogICAgaWYgb3MuZ2V0cHBpZCgpICE9IHBhcmVudF9waWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdQQVJFTlRfQ0hBTkdFRCcpCgoKZGVmIHNlYWwocm9vdCwgcGFyZW50X3BpZD1Ob25lKToKICAgIHF1YWxpZnkoKQogICAgcGFyZW50X3BpZCA9IG9zLmdldHBwaWQoKSBpZiBwYXJlbnRfcGlkIGlzIE5vbmUgZWxzZSBwYXJlbnRfcGlkCiAgICBhcm1fcGFyZW50X2RlYXRoKHBhcmVudF9waWQpCiAgICAjIFRoaXMgZnJlc2ggZXhlYyBoYXMgcmVjZWl2ZWQgbm8gYXNzZXNzb3IgZGF0YTsgcHJlLW9wZW5lZCBjb2RlL2FyZ3Mgb25seS4KICAgICMgQ2xvc2UgZXZlcnkgZXh0cmEgZGVzY3JpcHRvciwgbm90IG1lcmVseSB0aG9zZSBiZWxvdyBhIGxvd2VyZWQgcmxpbWl0LgogICAgZmRzID0gW2ludChwLm5hbWUpIGZvciBwIGluIFBhdGgoJy9wcm9jL3NlbGYvZmQnKS5pdGVyZGlyKCldCiAgICBudWxsID0gb3Mub3BlbignL2Rldi9udWxsJywgb3MuT19SRE9OTFkpCiAgICBvcy5kdXAyKG51bGwsIDApCiAgICBvcy5jbG9zZShudWxsKQogICAgZm9yIGZkIGluIGZkczoKICAgICAgICBpZiBmZCA+IDI6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG9zLmNsb3NlKGZkKQogICAgICAgICAgICBleGNlcHQgT1NFcnJvciBhcyBlcnJvcjoKICAgICAgICAgICAgICAgIGlmIGVycm9yLmVycm5vICE9IGVycm5vLkVCQURGOgogICAgICAgICAgICAgICAgICAgIHJhaXNlCiAgICBvcy5lbnZpcm9uLmNsZWFyKCkKICAgIG9zLmVudmlyb24udXBkYXRlKHsnTEFORyc6ICdDJywgJ1BZVEhPTkRPTlRXUklURUJZVEVDT0RFJzogJzEnfSkKICAgIG9zLmNocm9vdChyb290KQogICAgb3MuY2hkaXIoJy8nKQogICAgc3lzLnBhdGhbOl0gPSBbJy9ydW50aW1lJ10KICAgIHN5cy5hcmd2WzpdID0gWydjYW5kaWRhdGUnXQogICAgIyBEcm9wIGFtYmllbnQgYW5kIEVWRVJZIHN1cHBvcnRlZCBib3VuZGluZyBjYXBhYmlsaXR5IHdoaWxlIHByaXZpbGVnZWQuCiAgICBwcmN0bCg0NywgNCkKICAgIGNhcCA9IDAKICAgIHdoaWxlIFRydWU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcmN0bCgyMywgY2FwKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICBpZiBlcnJvci5lcnJubyA9PSBlcnJuby5FSU5WQUw6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICByYWlzZQogICAgICAgIHByY3RsKDI0LCBjYXApCiAgICAgICAgaWYgcHJjdGwoMjMsIGNhcCkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdCT1VORElOR19DQVBfUkVNQUlOUycpCiAgICAgICAgaWYgcHJjdGwoNDcsIDEsIGNhcCkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdBTUJJRU5UX0NBUF9SRU1BSU5TJykKICAgICAgICBjYXAgKz0gMQogICAgICAgIGlmIGNhcCA+IDEyODoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdVTktOT1dOX0NBUF9SQU5HRScpCiAgICBvcy5zZXRncm91cHMoW10pCiAgICBvcy5zZXRyZXNnaWQoNjU1MzQsIDY1NTM0LCA2NTUzNCkKICAgIG9zLnNldHJlc3VpZCg2NTUzNCwgNjU1MzQsIDY1NTM0KQogICAgaGVhZGVyID0gQ2FwSGVhZGVyKDB4MjAwODA1MjIsIDApCiAgICBkYXRhID0gKENhcERhdGEgKiAyKSgpCiAgICBpZiBMSUJDLmNhcHNldChjdHlwZXMuYnlyZWYoaGVhZGVyKSwgZGF0YSkgIT0gMDoKICAgICAgICByYWlzZSBPU0Vycm9yKGN0eXBlcy5nZXRfZXJybm8oKSwgJ2NhcHNldCcpCiAgICBpZiBMSUJDLmNhcGdldChjdHlwZXMuYnlyZWYoaGVhZGVyKSwgZGF0YSkgIT0gMCBvciBhbnkoeC5lZmZlY3RpdmUgb3IgeC5wZXJtaXR0ZWQgb3IgeC5pbmhlcml0YWJsZSBmb3IgeCBpbiBkYXRhKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NBUFNfUkVNQUlOJykKICAgIGlmIG9zLmdldHJlc3VpZCgpICE9ICg2NTUzNCwpKjMgb3Igb3MuZ2V0cmVzZ2lkKCkgIT0gKDY1NTM0LCkqMyBvciBvcy5nZXRncm91cHMoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0lERU5USVRZX1JFTUFJTlMnKQogICAgIyBJZGVudGl0eSB0cmFuc2l0aW9ucyBjbGVhciBQREVBVEhTSUcuIFJlLWFybSBhbmQgdmVyaWZ5IHRoZSBvcmlnaW5hbAogICAgIyBwYXJlbnQgaXMgc3RpbGwgYWxpdmUgYmVmb3JlIHRoZSBjYW5kaWRhdGUgY2FuIHJ1bi4KICAgIGFybV9wYXJlbnRfZGVhdGgocGFyZW50X3BpZCkKICAgIGZvciBuYW1lLCB2YWx1ZSBpbiBbKCdSTElNSVRfQVMnLCAyNjg0MzU0NTYpLCAoJ1JMSU1JVF9DUFUnLCAyKSwKICAgICAgICAgICAgICAgICAgICAgICAgKCdSTElNSVRfRlNJWkUnLCA2NTUzNiksICgnUkxJTUlUX0NPUkUnLCAwKSwKICAgICAgICAgICAgICAgICAgICAgICAgKCdSTElNSVRfTk9GSUxFJywgMzIpLCAoJ1JMSU1JVF9OUFJPQycsIDApXToKICAgICAgICByZXNvdXJjZS5zZXRybGltaXQoZ2V0YXR0cihyZXNvdXJjZSwgbmFtZSksICh2YWx1ZSwgdmFsdWUpKQogICAgcmVjZWlwdCA9IHsndWlkJzogbGlzdChvcy5nZXRyZXN1aWQoKSksICdnaWQnOiBsaXN0KG9zLmdldHJlc2dpZCgpKSwKICAgICAgICAgICAgICAgJ2dyb3Vwcyc6IG9zLmdldGdyb3VwcygpLCAnY2FwYWJpbGl0aWVzX3plcm8nOiBUcnVlLAogICAgICAgICAgICAgICAnYm91bmRpbmdfZHJvcHBlZCc6IGNhcCwgJ2FtYmllbnRfY2xlYXJlZCc6IFRydWUsCiAgICAgICAgICAgICAgICdwYXJlbnRfZGVhdGhfc2lnbmFsJzogc2lnbmFsLlNJR0tJTEwsCiAgICAgICAgICAgICAgICdjd2QnOiBvcy5nZXRjd2QoKSwgJ25vX25ld19wcml2cyc6IFRydWUsICdzZWNjb21wJzogJ25hdGl2ZS1hbGxvd2xpc3QnfQogICAgc2VjY29tcCgpCiAgICAjIFRydXN0ZWQgc3RhcnR1cCByZWNlaXB0IHByZWNlZGVzIHRoZSB1bnRydXN0ZWQgY29kZS4gQXNzZXNzb3IgbmV2ZXIgdHJlYXRzCiAgICAjIGFueSBsYXRlciBzdGRvdXQgYXMgYSB0cnVzdGVkIHZlcmRpY3Q7IG91dHB1dHMgcmVxdWlyZSBleGFjdCBjYXNlIG1hdGNoZXMuCiAgICBvcy53cml0ZSgyLCAoanNvbi5kdW1wcyh7J3NlYWxlZCc6IHJlY2VpcHR9KSsnXG4nKS5lbmNvZGUoKSkKCmRlZiBtYWluKCk6CiAgICBwYXJlbnRfcGlkID0gb3MuZ2V0cHBpZCgpCiAgICBhcm1fcGFyZW50X2RlYXRoKHBhcmVudF9waWQpCiAgICBpZiBsZW4oc3lzLmFyZ3YpICE9IDI6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdPTkVfUk9PVF9SRVFVSVJFRCcpCiAgICByb290ID0gUGF0aChzeXMuYXJndlsxXSkucmVzb2x2ZShzdHJpY3Q9VHJ1ZSkKICAgIHNvdXJjZSA9IChyb290Lyd3b3JrL2NhbmRpZGF0ZS5weScpLnJlYWRfdGV4dCgpCiAgICByZXF1ZXN0ID0ganNvbi5sb2Fkcygocm9vdC8nd29yay9hcmdzLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGlmIHNldChyZXF1ZXN0KSAhPSB7J21vZGUnLCAnYXJncyd9IG9yIHJlcXVlc3RbJ21vZGUnXSBub3QgaW4gKCdjYW5kaWRhdGUnLCAnaW5zcGVjdGlvbicpIG9yIG5vdCBpc2luc3RhbmNlKHJlcXVlc3RbJ2FyZ3MnXSwgbGlzdCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdCQURfUFJPVE9DT0wnKQogICAgc2VhbChzdHIocm9vdCksIHBhcmVudF9waWQpCiAgICBjb2RlID0gY29tcGlsZShzb3VyY2UsICcvd29yay9jYW5kaWRhdGUucHknLCAnZXhlYycpCiAgICAjIFNvdXJjZS9yZXF1ZXN0IGNvbnRhaW4gb25seSB0aGUgY3VycmVudCBjYWxsLCBuZXZlciB0aGUgYXNzZXNzb3IgZml4dHVyZS4KICAgIHNjb3BlID0geydfX25hbWVfXyc6ICdfX2NhbmRpZGF0ZV9fJ30KICAgIGV4ZWMoY29kZSwgc2NvcGUsIHNjb3BlKQogICAgdmFsdWUgPSBzY29wZVsncnVuJ10oKnJlcXVlc3RbJ2FyZ3MnXSkKICAgIHByaW50KGpzb24uZHVtcHModmFsdWUsIGFsbG93X25hbj1GYWxzZSksIGZsdXNoPVRydWUpCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigpCg=='}, 'probe.py': {'sha256': '610aef51200e5587db0ba11595d8c1c5f06f2e711536ed1fd7c53e0d90c0a58f', 'base64': '...'}}
with tempfile.TemporaryDirectory(prefix='containment-package-') as temp:
    root = Path(temp)
    for name, item in assets.items():
        data = base64.b64decode(item['base64'])
        if hashlib.sha256(data).hexdigest() != item['sha256']:
            raise RuntimeError('ASSET_HASH_MISMATCH')
        (root/name).write_bytes(data)
    process = subprocess.run([sys.executable, '-B', str(root/'probe.py'), '/kaggle/working/qualification'],
                             stdin=subprocess.DEVNULL, close_fds=True, capture_output=True,
                             text=True, timeout=180, env={'PATH':'/usr/local/bin:/usr/bin:/bin','LANG':'C'})
    print('PROBE_RETURN_CODE', process.returncode)
    print(process.stdout[:16384])
    print(process.stderr[:16384])
    # Notebook COMPLETE is not qualification PASS. Preserve unsupported receipts.
    result = Path('/kaggle/working/qualification/result.json')
    if not result.exists():
        raise RuntimeError('MISSING_QUALIFICATION_RECEIPT')
